# 07 — NHANES: download, clean, split (the dataset with real ground truth)

**Why NHANES.** NHANES asks people the same question the CDC survey asks ("Has a doctor ever told you
that you have diabetes?") **and** measures their blood (HbA1c). So for every adult we have both:

* **Ỹ, the recorded label** — the self-reported diagnosis (what a dataset like the CDC survey contains);
* **Y, the true status** — diagnosed, *or* HbA1c >= 6.5% (the standard laboratory cut-off for diabetes).

A **missed event** is Y = 1, Ỹ = 0: a person with diabetes by blood test who has never been told. These
are real label errors, so nothing is injected. The discovery methods only ever see Ỹ; Y is used in
notebook 08 *after* every method has made its picks, to check who found the right groups.

| Step | Input | Output |
|---|---|---|
| 1 Download | CDC NHANES website, 7 cycles 2005-2018 | `data/nhanes_raw/*.xpt` (skips files already there) |
| 2 Merge | the .xpt files | one row per person per cycle |
| 3 Variables + cohort | merged frame | `results/07_nhanes_cleaning_log.csv` |
| 4 Labels | HbA1c + self-report | counts only (no group breakdown here — that would be peeking) |
| 5 Split 60/20/20 | clean frame | `data/nhanes_{train,val,test}.csv`, `data/nhanes_clean.csv` |

**Cycles.** 2005-06 through 2017-18. Later cycles are left out: 2019-20 was cut short by the pandemic, and
2021-23 dropped the "when was your cholesterol last checked" question, which this study needs.

**Variable sources (CDC documentation pages):** DEMO, DIQ (DIQ010 diagnosis), GHB (LBXGH HbA1c), BMX (BMXBMI), BPQ (BPQ020, BPQ060, BPQ070, BPQ080), HIQ (HIQ011 insurance),
HUQ (HUQ010 general health, HUQ030 routine place for care, HUQ050/HUQ051 visits), SMQ (SMQ020), MCQ
(MCQ160C/E/F heart disease, heart attack, stroke).

## 0. Settings

In [ ]:
import io, time, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import li_lib as li

CYCLES = {"D": 2005, "E": 2007, "F": 2009, "G": 2011, "H": 2013, "I": 2015, "J": 2017}   # suffix -> folder year
FILES = ["DEMO", "DIQ", "GHB", "BMX", "BPQ", "HIQ", "HUQ", "SMQ", "MCQ"]
URL = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/{year}/DataFiles/{file}_{suf}.xpt"
RAW = li.DATA / "nhanes_raw"; RAW.mkdir(exist_ok=True)
SEED = 20261020
A1C_CUTOFF = 6.5      # % — standard diagnostic threshold
MIN_AGE = 20          # adults, as in the CDC's own undiagnosed-diabetes estimates
print("saving raw files to", RAW)

## 1. Download (about 63 small files; skips anything already downloaded)
If a download fails, open the printed URL in your browser, save the file into `data/nhanes_raw/` with the
same name, and rerun this cell.

In [ ]:
failed = []
for suf, year in CYCLES.items():
    for f in FILES:
        dest = RAW / f"{f}_{suf}.xpt"
        if dest.exists() and dest.stat().st_size > 1000:
            continue
        url = URL.format(year=year, file=f, suf=suf)
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            data = urllib.request.urlopen(req, timeout=120).read()
            if not data[:80].startswith(b"HEADER RECORD"):
                raise ValueError("not an XPT file (the site returned a web page)")
            dest.write_bytes(data); print(f"ok   {dest.name}  {len(data)/1e6:.1f} MB")
            time.sleep(0.5)
        except Exception as e:
            failed.append(url); print(f"FAIL {url}  ({e})")
print("all files present" if not failed else f"{len(failed)} files failed — see above")

## 2. Merge the files within each cycle (on SEQN, the person ID), then stack the cycles
SAS transport files store the value 0 as a tiny number (5.4e-79); it is reset to exactly 0 on reading,
otherwise "0 doctor visits" would never equal 0.

In [ ]:
frames = []
for suf in CYCLES:
    d = None
    for f in FILES:
        x = pd.read_sas(RAW / f"{f}_{suf}.xpt", format="xport")
        num = x.select_dtypes("number").columns
        x[num] = x[num].mask(x[num].abs() < 1e-10, 0.0)   # SAS transport files store 0 as 5.4e-79
        d = x if d is None else d.merge(x, on="SEQN", how="left", suffixes=("", f"_{f}"))
    d["cycle"] = suf
    frames.append(d)
raw = pd.concat(frames, ignore_index=True, sort=False)
print(f"{len(raw):,} people across {len(CYCLES)} cycles")
print(raw.groupby("cycle").size().rename("people").to_frame().T)

## 3. Build the variables (CDC-survey names where the meaning matches) and the adult cohort
Answers coded 7 (refused) or 9 (don't know) become missing. **HbA1c is never a model feature** — it is the
ground truth.

**Left out on purpose: DIQ180** (blood sugar test in past 3 years). It would be the ideal recording variable,
but NHANES only asks it of people who were *not* diagnosed (diagnosed people skip it). Its missing values
would therefore give away the label. CholCheck is asked of everyone, so it is the recording variable here.

| Variable | Meaning | Source |
|---|---|---|
| HighBP | ever told high blood pressure | BPQ020 = 1 |
| HighChol | ever told high cholesterol (0 if never checked) | BPQ080 = 1 |
| **CholCheck** | cholesterol checked within 5 years (*recording variable*) | BPQ060, BPQ070 |
# | AnyHealthcare | has health insurance | HIQ011 = 1 |
| RoutineCare | has a usual place for care | HUQ030 = 1 or 3 |
| VisitedDoctor | any doctor visit in past 12 months | HUQ050/HUQ051 != 0 |
| BMI, Smoker, Stroke, HeartDiseaseorAttack, GenHlth | as in the CDC survey | BMX, SMQ, MCQ, HUQ010 |
| Sex (1 = male), Age, Education, Income | Income = family income-to-poverty ratio | DEMO |
| race_black, race_hispanic | protected attributes | RIDRETH1 |

In [ ]:
def yes(col, codes=(1,)):
    v = raw[col] if col in raw else pd.Series(np.nan, index=raw.index)
    out = v.isin(codes).astype(float)
    out[v.isna() | v.isin([7, 9, 77, 99])] = np.nan
    return out

def coalesce(*cols):
    """Same question under different names in different cycles (e.g. HUQ050 until 2016, HUQ051 from 2017)."""
    out = pd.Series(np.nan, index=raw.index)
    for c in cols:
        if c in raw:
            out = out.fillna(raw[c])
    return out

d = pd.DataFrame({"SEQN": raw["SEQN"], "cycle": raw["cycle"]})
d["HighBP"] = yes("BPQ020")
d["HighChol"] = yes("BPQ080"); d.loc[raw["BPQ060"] == 2, "HighChol"] = 0.0      # never checked -> never told
chk = pd.Series(np.nan, index=raw.index)
chk[raw["BPQ070"].isin([1, 2, 3])] = 1.0
chk[raw["BPQ070"].eq(4) | raw["BPQ060"].eq(2)] = 0.0
d["CholCheck"] = chk
d["AnyHealthcare"] = yes("HIQ011")
d["RoutineCare"] = yes("HUQ030", codes=(1, 3))
visits = coalesce("HUQ050", "HUQ051")
d["VisitedDoctor"] = np.where(visits.isna() | visits.isin([77, 99]), np.nan, (visits != 0).astype(float))
d["BMI"] = raw["BMXBMI"]
d["Smoker"] = yes("SMQ020")
d["Stroke"] = yes("MCQ160F")
d["HeartDiseaseorAttack"] = np.fmax(yes("MCQ160C"), yes("MCQ160E"))
d["GenHlth"] = raw["HUQ010"].where(raw["HUQ010"].between(1, 5))
d["Sex"] = (raw["RIAGENDR"] == 1).astype(float)
d["Age"] = raw["RIDAGEYR"]
d["Education"] = raw["DMDEDUC2"].where(raw["DMDEDUC2"].between(1, 5))
d["Income"] = raw["INDFMPIR"]
d["race_black"] = (raw["RIDRETH1"] == 4).astype(float)
d["race_hispanic"] = raw["RIDRETH1"].isin([1, 2]).astype(float)
d["a1c"] = raw["LBXGH"]
d["dx_code"] = raw["DIQ010"]
d["pregnant"] = raw["RIDEXPRG"].eq(1) if "RIDEXPRG" in raw else False

log = [("all people, 7 cycles", len(d))]
d = d[d["Age"] >= MIN_AGE]; log.append((f"age >= {MIN_AGE}", len(d)))
d = d[~d["pregnant"].astype(bool)]; log.append(("not pregnant", len(d)))
d = d[d["dx_code"].isin([1, 2, 3])]; log.append(("answered the diabetes question", len(d)))
d = d[d["a1c"].notna()]; log.append(("has an HbA1c lab result", len(d)))
FEATURES = ["HighBP", "HighChol", "CholCheck", "AnyHealthcare", "RoutineCare", "VisitedDoctor",
            "BMI", "Smoker", "Stroke", "HeartDiseaseorAttack", "GenHlth", "Sex", "Age", "Education", "Income",
            "race_black", "race_hispanic"]
print("missing values per feature (before dropping):")
print(d[FEATURES].isna().sum().sort_values(ascending=False).head(8).to_string())
d = d.dropna(subset=FEATURES); log.append(("no missing feature values", len(d)))
const = [c for c in FEATURES if d[c].nunique() < 2]
assert not const, f"constant feature(s) {const}: check the coding above before going on"
log = pd.DataFrame(log, columns=["step", "rows_remaining"])
li.save(log, "07_nhanes_cleaning_log.csv", show=10)

## 4. Labels
* `diabetes_recorded` (Ỹ) = told by a doctor (DIQ010 = 1). "Borderline" counts as not told.
* `diabetes_true` (Y) = told, **or** HbA1c >= 6.5%.
* `missed` = true but not recorded.

A single HbA1c is a screening-grade truth: some people near 6.5% would not be confirmed on a second test.
The CDC's own estimates also use fasting glucose; we use HbA1c alone so that every adult has a truth value.
Only overall counts are shown here; how the missed cases split across groups is looked at in notebook 08,
after the methods have made their picks.

In [ ]:
d["diabetes_recorded"] = (d["dx_code"] == 1).astype(int)
d["diabetes_true"] = ((d["dx_code"] == 1) | (d["a1c"] >= A1C_CUTOFF)).astype(int)
d["missed"] = ((d["diabetes_true"] == 1) & (d["diabetes_recorded"] == 0)).astype(int)
summary = pd.DataFrame({
    "adults": [len(d)],
    "recorded_rate": [d.diabetes_recorded.mean()],
    "true_rate": [d.diabetes_true.mean()],
    "missed_among_true": [d.missed.sum() / d.diabetes_true.sum()],
    "missed_n": [int(d.missed.sum())]})
print(summary.round(4).to_string(index=False))
print(d.groupby("cycle")[["diabetes_recorded", "diabetes_true", "missed"]].mean().round(4))

## 5. Split 60 / 20 / 20, stratified on the recorded label only (no group is known in advance)

In [ ]:
from sklearn.model_selection import train_test_split
keep_cols = ["SEQN", "cycle"] + FEATURES + ["diabetes_recorded", "diabetes_true", "missed", "a1c"]
d = d[keep_cols].reset_index(drop=True)
train, rest = train_test_split(d, test_size=0.4, random_state=SEED, stratify=d["diabetes_recorded"])
val, test = train_test_split(rest, test_size=0.5, random_state=SEED, stratify=rest["diabetes_recorded"])
d.to_csv(li.DATA / "nhanes_clean.csv", index=False)
for name, part in [("train", train), ("val", val), ("test", test)]:
    part.reset_index(drop=True).to_csv(li.DATA / f"nhanes_{name}.csv", index=False)
print(pd.DataFrame({"rows": [len(train), len(val), len(test)],
                    "recorded_rate": [p.diabetes_recorded.mean() for p in (train, val, test)],
                    "true_rate": [p.diabetes_true.mean() for p in (train, val, test)]},
                   index=["train", "val", "test"]).round(4))